# Analysis and Charts

Regenerates the five CSV exports from the per-cell JSON results, then loads the CSVs and produces every table and figure in the paper. Figures are written to `figures/`.

This notebook **calls the scripts in `scripts/`** as subprocesses; it does not re-implement the analysis logic. The same scripts are used by the main experiment repo to produce the same numbers.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR = REPO_ROOT / 'data'
SCRIPTS_DIR = REPO_ROOT / 'scripts'
FIGURES_DIR = REPO_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

for d in ('9b','27b','30b','gemma4','qwen36','opus47'):
    assert (RESULTS_DIR / d).exists(), f'Missing {RESULTS_DIR / d}'
print(f'REPO_ROOT = {REPO_ROOT}')
print(f'Six lane dirs present in {RESULTS_DIR}/')

## 1. Regenerate the five CSV exports

These analysis scripts write into the local `data/` directory.

In [ ]:
# build_results_csv.py expects relative paths; run it from REPO_ROOT and pass absolute paths.
def run_script(args, cwd=REPO_ROOT):
    print('$', ' '.join(map(str, args)))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError(f'Script failed: {args}')
    print(result.stdout)
    print(result.stderr)

run_script([sys.executable, str(SCRIPTS_DIR / 'build_results_csv.py'),
            '--results-dirs',
            *[str(RESULTS_DIR / f'{m}') for m in ('9b','27b','30b','gemma4','qwen36','opus47')],
            '--output', str(DATA_DIR / 'all-results.csv')])

In [ ]:
# Regenerate the taxonomy + per-question-stats + semantic-error CSVs from the per-cell JSONs.
# All resolve REPO_ROOT = scripts/.. and read the BIRD source data from
# bird_mini_dev/minidev/MINIDEV/ (downloaded by notebook 01) + the six lane dirs in results/,
# so they run standalone in the public flat layout (no main-repo paths needed).
run_script([sys.executable, str(SCRIPTS_DIR / 'question_taxonomy.py')])
run_script([sys.executable, str(SCRIPTS_DIR / 'per_question_stats.py'),
            '--csv', str(DATA_DIR / 'per-question-stats.csv')])
run_script([sys.executable, str(SCRIPTS_DIR / 'semantic_error_taxonomy.py')])

for f in ('all-results.csv', 'question-taxonomy.csv', 'per-question-stats.csv',
          'status-budget.csv', 'semantic-errors.csv'):
    p = DATA_DIR / f
    print(f'  {f}: {"OK" if p.exists() else "MISSING"} ({p.stat().st_size if p.exists() else 0} bytes)')

In [ ]:
run_script([sys.executable, str(SCRIPTS_DIR / 'status_budget_table.py'),
            *[str(RESULTS_DIR / f'{m}') for m in ('9b','27b','30b','gemma4','qwen36','opus47')],
            '--csv', str(DATA_DIR / 'status-budget.csv')])

## 2. Load the CSVs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='paper', font_scale=1.0)
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.dpi'] = 100

results = pd.read_csv(DATA_DIR / 'all-results.csv')
stats   = pd.read_csv(DATA_DIR / 'per-question-stats.csv')
taxonomy = pd.read_csv(DATA_DIR / 'question-taxonomy.csv')
status   = pd.read_csv(DATA_DIR / 'status-budget.csv')

MODEL_ORDER = ['9B','27B','30B','QWEN36','GEMMA4','OPUS47']
MODEL_LABEL = {'9B':'Qwen3.5-9B','27B':'Qwen3.5-27B','30B':'Qwen3-Coder-30B',
               'QWEN36':'Qwen3.6-35B-A3B','GEMMA4':'gemma-4-26b','OPUS47':'claude-opus-4-7'}
HF_MODELS = {'9B','27B','30B'}
SCOPED_MODELS = {'QWEN36','GEMMA4','OPUS47'}

print('results:',  results.shape)
print('stats:',    stats.shape)
print('taxonomy:', taxonomy.shape)
print('status:',   status.shape)

## 3. Figure 1: Headline delta BEX (native L4 vs L0-PAD)

In [ ]:
def model_delta(df, m):
    g = df[df.model == m]
    l0p = g[g.level == 'L0-PAD']['bex_pct'].mean()
    l4 = g[g.level == 'L4']['bex_pct'].mean()
    return l0p, l4, l4 - l0p

rows = [(m, *model_delta(results, m)) for m in MODEL_ORDER]
deltas = pd.DataFrame(rows, columns=['model','L0PAD','L4','delta_pp'])
print(deltas.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
labels = [MODEL_LABEL[m] for m in deltas['model']]
bars = ax.bar(labels, deltas['delta_pp'], color='#3a7ca5')
ax.axhspan(15, 20, color='#cce5ff', alpha=0.35, label='+15 to +20 pp band')
ax.set_ylabel('delta BEX (percentage points)')
ax.set_title('Headline: native L4 vs L0-PAD delta BEX, six-model panel')
for b, v in zip(bars, deltas['delta_pp']):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3, f'{v:+.1f}', ha='center', fontsize=9)
ax.set_ylim(0, max(deltas['delta_pp'])*1.18)
plt.xticks(rotation=20, ha='right')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig01-headline-delta-bex.png')
plt.show()

## 4. Figure 2: Per-model x per-level ER / BEX heatmap

In [ ]:
pivot_bex = (results.assign(model=pd.Categorical(results.model, categories=MODEL_ORDER, ordered=True))
             .pivot_table(index='model', columns='level', values='bex_pct'))
level_order = ['L0','L0-PAD','L4','L4-DD','L4-QP','L4-BC','L4-DK','EVIDENCE']
pivot_bex = pivot_bex.reindex(columns=[c for c in level_order if c in pivot_bex.columns])

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.heatmap(pivot_bex, annot=True, fmt='.1f', cmap='viridis', cbar_kws={'label':'BEX %'}, ax=ax)
ax.set_title('BEX% by model x level (mean across 11 BIRD databases)')
ax.set_xlabel('Level'); ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig02-bex-heatmap.png')
plt.show()

## 5. Figure 3: ER-BEX gap at native L4

In [ ]:
def l4_gap(df, m):
    g = df[df.model == m]
    return g[g.level == 'L4']['gap_pp'].mean()

gaps = pd.DataFrame([(m, l4_gap(results, m)) for m in MODEL_ORDER], columns=['model','gap_pp'])
print(gaps.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar([MODEL_LABEL[m] for m in gaps.model], gaps.gap_pp, color='#e07a5f')
ax.set_ylabel('ER-BEX gap at L4 (pp)')
ax.set_title('ER-BEX gap at native L4, six-model panel')
for i, v in enumerate(gaps.gap_pp):
    ax.text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig03-er-bex-gap.png')
plt.show()

## 6. Figure 4: Leave-one-out ablation (HF lanes)

In [ ]:
loo_levels = ['L4-DD','L4-QP','L4-BC','L4-DK']
loo_df = results[results.model.isin(HF_MODELS) & results.level.isin(loo_levels)].copy()
loo_df['model'] = pd.Categorical(loo_df['model'], categories=['9B','27B','30B'], ordered=True)
loo = loo_df.pivot_table(index='model', columns='level', values='er_pct', observed=False).reindex(columns=loo_levels)

fig, ax = plt.subplots(figsize=(8, 4.2))
loo.plot(kind='bar', ax=ax, color=['#84a59d','#f28482','#84a59d','#84a59d'])
ax.set_ylabel('ER %')
ax.set_title('Leave-one-out ablation, ER% (Dimensional Lanes). Dropping QP hurts every model the most.')
ax.set_xticklabels([MODEL_LABEL[m.get_text()] for m in ax.get_xticklabels()], rotation=0)
ax.legend(title='dimension dropped', loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig04-loo-ablation.png')
plt.show()

## 7. Figure 5: Per-database ER panel with hostile databases highlighted

In [ ]:
def l4_ex(df, m):
    g = df[df.model == m]
    return g[g.level == 'L4'][['database','er_pct']]

per_db = pd.concat([l4_ex(results, m).assign(model=m) for m in MODEL_ORDER], ignore_index=True)
pivot = per_db.pivot_table(index='database', columns='model', values='er_pct').reindex(columns=MODEL_ORDER)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label':'ER %'}, ax=ax)
ax.set_title('Per-database ER% at native L4, six models. Low-ER: thrombosis_prediction, european_football_2, student_club')
ax.set_xticklabels([MODEL_LABEL[c.get_text()] for c in ax.get_xticklabels()], rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig05-per-db-er.png')
plt.show()

## 8. Figure 6: student_club QP-load-bearing swing

In [ ]:
sc = results[(results.database == 'student_club') & results.model.isin(['9B','27B','30B'])
             & results.level.isin(['L4','L4-QP'])]
sc_pivot = sc.pivot_table(index='model', columns='level', values='er_pct').reindex(['9B','27B','30B'])
print(sc_pivot.to_string())
fig, ax = plt.subplots(figsize=(7, 4.0))
sc_pivot.plot(kind='bar', ax=ax, color=['#84a59d','#f28482'])
ax.set_ylabel('ER % on student_club')
ax.set_title('student_club: dropping QP from L4 collapses ER on every Dimensional-lane model')
ax.set_xticklabels([MODEL_LABEL[m] for m in ['9B','27B','30B']], rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig06-student-club-qp.png')
plt.show()

## 9. Figure 7: Bootstrap CI forest plot

In [ ]:
# stats CSV is in long form. Extract bootstrap_question and bootstrap_database rows.
def extract_ci(df, table_name):
    rows = []
    for model in MODEL_ORDER:
        sub = df[(df.table == table_name) & (df.model == model)]
        if sub.empty:
            continue
        pe = float(sub[sub.metric == 'point_estimate_pp'].value.iloc[0])
        lo = float(sub[sub.metric == 'ci_lo_pp'].value.iloc[0])
        hi = float(sub[sub.metric == 'ci_hi_pp'].value.iloc[0])
        rows.append((model, pe, lo, hi))
    return pd.DataFrame(rows, columns=['model','pe','lo','hi'])

ci_q = extract_ci(stats, 'bootstrap_question')
ci_d = extract_ci(stats, 'bootstrap_database')

fig, ax = plt.subplots(figsize=(8.5, 4.5))
y = np.arange(len(MODEL_ORDER))
ax.errorbar(ci_q.pe, y - 0.15, xerr=[ci_q.pe - ci_q.lo, ci_q.hi - ci_q.pe], fmt='o',
            color='#3a7ca5', label='question-level 95% CI', capsize=4)
ax.errorbar(ci_d.pe, y + 0.15, xerr=[ci_d.pe - ci_d.lo, ci_d.hi - ci_d.pe], fmt='s',
            color='#e07a5f', label='database-level 95% CI', capsize=4)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.7)
ax.set_yticks(y); ax.set_yticklabels([MODEL_LABEL[m] for m in MODEL_ORDER])
ax.set_xlabel('delta BEX (pp)')
ax.set_title('Bootstrap 95% CIs on delta BEX, two resampling units')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig07-bootstrap-cis.png')
plt.show()

## 10. Figure 8: Stratified BEX lift by question type

In [ ]:
type_dist = taxonomy['question_type_primary'].value_counts().reindex(
    ['lookup','single-join','multi-join','aggregation','nested','case-when','cte','set-op','window'])
type_order = type_dist.dropna().index.tolist()

rows = []
for t in type_order:
    sub = taxonomy[taxonomy['question_type_primary'] == t]
    # Native L0 and native L4 for all six models.
    all_codes = ('9b','27b','30b','gemma4','qwen36','opus47')
    l0_cols = [f'{m}_L0_bex' for m in all_codes]
    l4_cols = [f'{m}_L4_bex' for m in all_codes]
    l0_vals = sub[l0_cols].stack().dropna().astype(float)
    l4_vals = sub[l4_cols].stack().dropna().astype(float)
    if l0_vals.empty or l4_vals.empty:
        continue
    rows.append((t, len(l0_vals)+len(l4_vals), 100*l0_vals.mean(), 100*l4_vals.mean()))
by_type = pd.DataFrame(rows, columns=['type','n','L0_bex','L4_bex'])
by_type['delta'] = by_type['L4_bex'] - by_type['L0_bex']
print(by_type.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(by_type))
w = 0.4
ax.bar(x - w/2, by_type.L0_bex, w, label='L0', color='#cccccc')
ax.bar(x + w/2, by_type.L4_bex, w, label='L4', color='#3a7ca5')
ax.set_xticks(x); ax.set_xticklabels(by_type.type, rotation=20, ha='right')
ax.set_ylabel('BEX %')
ax.set_title('BEX by question type: where does metadata help most?')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig08-by-question-type.png')
plt.show()

## 11. Figure 9: Operational status budget

In [ ]:
status['round'] = status['round'].str.replace('','', regex=False).str.upper()
status = status.rename(columns={'round': 'model'})
totals = status.groupby('model')[['n','ok','execution_error','parse_error','empty_response_token_exhaustion']].sum()
totals = totals.reindex(['9B','27B','30B','QWEN36','GEMMA4','OPUS47'])
pct = totals.div(totals['n'], axis=0).drop(columns=['n']) * 100

fig, ax = plt.subplots(figsize=(9, 4.5))
bottom = np.zeros(len(pct))
colors = {'ok':'#84a59d','execution_error':'#e07a5f','parse_error':'#f6bd60','empty_response_token_exhaustion':'#9b5de5'}
for col in ['ok','execution_error','parse_error','empty_response_token_exhaustion']:
    ax.bar([MODEL_LABEL[m] for m in pct.index], pct[col], bottom=bottom, label=col, color=colors[col])
    bottom += pct[col].values
ax.set_ylabel('% of cases')
ax.set_title('Operational status budget per model')
ax.legend(loc='lower right', framealpha=0.9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig09-status-budget.png')
plt.show()

## 12. Figure 10: BEX scatter, native L4 vs L0-PAD baseline

Each model is one labeled point. Dimensional lanes (9B, 27B, 30B) are drawn as circles; Cross-vendor lanes (Qwen3.6-35B-A3B, gemma-4-26b, claude-opus-4-7) as triangles. The diagonal y = x shows the no-effect line; the visible vertical lift of every point above the diagonal is the metadata effect.

In [ ]:
rows = []
for m in MODEL_ORDER:
    g = results[results.model == m]
    l0p = g[g.level == 'L0-PAD']['bex_pct'].mean()
    l4 = g[g.level == 'L4']['bex_pct'].mean()
    rows.append({'model': m, 'L0PAD': l0p, 'L4': l4,
                 'lane': 'HF' if m in HF_MODELS else 'Scoped'})
scat = pd.DataFrame(rows)
print(scat.to_string(index=False))

# Lane display names: HF = the three Qwen dimensional-ablation models; Scoped = the cross-vendor models.
LANE_LABEL = {'HF': 'Dimensional', 'Scoped': 'Cross-vendor'}

fig, ax = plt.subplots(figsize=(9, 5.5))
for lane, marker in (('HF', 'o'), ('Scoped', '^')):
    sub = scat[scat.lane == lane]
    ax.scatter(sub.L0PAD, sub.L4, marker=marker, s=110,
               edgecolors='black', linewidths=0.7, label=LANE_LABEL[lane],
               c=['#3a7ca5' if lane == 'HF' else '#e07a5f'] * len(sub))

# Per-model label placement (tuned to the final-run geometry): the 27B/30B pair sits
# almost on top of each other near (15, 34), so 27B goes up-left and 30B down-right.
# Thin leader lines keep each label tied to its marker.
label_pos = {  # model: (dx_pt, dy_pt, ha, va)
    '9B':     ( 8,   0, 'left',  'center'),
    '27B':    (-10,  9, 'right', 'bottom'),
    '30B':    ( 10, -9, 'left',  'top'),
    'QWEN36': ( 8,   0, 'left',  'center'),
    'GEMMA4': ( 8,   0, 'left',  'center'),
    'OPUS47': ( 8,   0, 'left',  'center'),
}
for _, r in scat.iterrows():
    dx, dy, ha, va = label_pos[r.model]
    ax.annotate(MODEL_LABEL[r.model], (r.L0PAD, r.L4),
                xytext=(dx, dy), textcoords='offset points',
                fontsize=9, ha=ha, va=va,
                arrowprops=dict(arrowstyle='-', color='0.6', lw=0.5, shrinkA=1, shrinkB=4))

lo, hi = 0, max(scat.L4.max(), scat.L0PAD.max()) + 8
ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('L0-PAD BEX (%)')
ax.set_ylabel('L4 BEX (%)')
ax.set_title('BEX: native L4 vs L0-PAD baseline (six-model panel)')
ax.legend(title='Lane', loc='lower right')
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.6)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig10-bex-scatter.png')
plt.show()

## 13. Figure 11: ER vs BEX scatter at native L4

Each model is one labeled point. The diagonal y = x is drawn for reference; the vertical distance from each point to the diagonal is that model's ER-BEX gap. Dimensional lanes appear as circles, Cross-vendor lanes as triangles.

In [ ]:
rows = []
for m in MODEL_ORDER:
    g = results[results.model == m]
    sub = g[g.level == 'L4']
    rows.append({'model': m, 'BEX': sub.bex_pct.mean(), 'ER': sub.er_pct.mean(),
                 'lane': 'HF' if m in HF_MODELS else 'Scoped'})
gap_df = pd.DataFrame(rows)
print(gap_df.to_string(index=False))

# Lane display names: HF = the three Qwen dimensional-ablation models; Scoped = the cross-vendor models.
LANE_LABEL = {'HF': 'Dimensional', 'Scoped': 'Cross-vendor'}

fig, ax = plt.subplots(figsize=(9, 5.5))
for lane, marker in (('HF', 'o'), ('Scoped', '^')):
    sub = gap_df[gap_df.lane == lane]
    ax.scatter(sub.BEX, sub.ER, marker=marker, s=110,
               edgecolors='black', linewidths=0.7, label=LANE_LABEL[lane],
               c=['#3a7ca5' if lane == 'HF' else '#e07a5f'] * len(sub))

# Per-model label placement (tuned to the final-run geometry): the upper cluster
# (27B/30B at BEX~34, Qwen36 ~37, opus ~44) is spread in different directions, and
# 9B's label is moved off its own marker. Thin leader lines tie each label to its point.
label_pos = {  # model: (dx_pt, dy_pt, ha, va)
    '9B':     (-10,  0, 'right',  'center'),
    'GEMMA4': (  0, -13, 'center', 'top'),
    '30B':    (  0, -13, 'center', 'top'),
    '27B':    (-10,  3, 'right',  'bottom'),
    'QWEN36': (  0,  13, 'center', 'bottom'),
    'OPUS47': ( 10,  0, 'left',   'center'),
}
for _, r in gap_df.iterrows():
    dx, dy, ha, va = label_pos[r.model]
    ax.annotate(MODEL_LABEL[r.model], (r.BEX, r.ER),
                xytext=(dx, dy), textcoords='offset points',
                fontsize=9, ha=ha, va=va,
                arrowprops=dict(arrowstyle='-', color='0.6', lw=0.5, shrinkA=1, shrinkB=4))

ax.plot([0, 100], [0, 100], color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='y = x')
ax.set_xlim(0, max(gap_df.BEX.max(), gap_df.ER.max()) + 12)
ax.set_ylim(0, 100)
ax.set_xlabel('BEX (%) at L4')
ax.set_ylabel('ER (%) at L4')
ax.set_title('ER vs BEX at native L4 (vertical distance from y=x is the ER-BEX gap)')
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.6)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig11-er-bex-gap-scatter.png')
plt.show()

## 15. Figure 12: Metadata lift across all four metrics (ER, Soft-F1, VM, BEX)

In [ ]:
# Figure 12: metadata lifts all four metrics (ER, Soft-F1, VM, BEX), not just BEX.
# Baseline = L0-PAD; treatment = native L4 for all six models.
# Soft-F1 is a 0-1 score shown x100 for visual comparability with the percentage metrics.
METRICS = [('er_pct', 'ER', 1.0), ('soft_f1', 'Soft-F1', 100.0),
           ('vm_pct', 'VM', 1.0), ('bex_pct', 'BEX', 1.0)]

def metric_macro(df, m, col, kind):
    g = df[df.model == m]
    sub = g[g.level == ('L0-PAD' if kind == 'base' else 'L4')]
    return sub[col].mean()

lift = pd.DataFrame(
    [(name,
      np.mean([metric_macro(results, m, col, 'base') for m in MODEL_ORDER]) * scale,
      np.mean([metric_macro(results, m, col, 'l4')   for m in MODEL_ORDER]) * scale)
     for col, name, scale in METRICS],
    columns=['metric', 'L0PAD', 'L4'])
lift['lift_pp'] = lift['L4'] - lift['L0PAD']
print(lift.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(lift))
w = 0.38
b0 = ax.bar(x - w/2, lift['L0PAD'], w, label='L0-PAD (distractor-text control)', color='#3a7ca5')
b4 = ax.bar(x + w/2, lift['L4'], w, label='L4 (full metadata)', color='#e07a5f')
ax.set_xticks(x)
ax.set_xticklabels(lift['metric'])
ax.set_ylabel('score (database-macro, six-model mean)')
ax.set_title('Metadata lifts all four metrics, not just BEX')
for bars in (b0, b4):
    for bar in bars:
        v = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5, f'{v:.1f}', ha='center', fontsize=8)
ax.set_ylim(0, max(lift['L4'].max(), lift['L0PAD'].max()) * 1.18)
ax.legend(loc='upper left')
ax.text(0.99, 0.97, 'Soft-F1 shown ×100', transform=ax.transAxes,
        ha='right', va='top', fontsize=7, style='italic', color='gray')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig12-four-metric-lift.png')
plt.show()

## 14. Summary

All figures are saved in `figures/`. The notebook regenerates **all five** CSV exports (`all-results`, `status-budget`, `question-taxonomy`, `per-question-stats`, `semantic-errors`) from the per-cell JSON results in `results/` plus the BIRD source data downloaded by notebook 01 - no pre-shipped CSVs required. Every chart used in the paper is reproducible end-to-end from this notebook (11 figures total).